# Step 4 EDA

Focused exploratory analysis for `outputs/statsbomb/penalties_statsbomb_clean.parquet`.

The reproducible script `reports/step4_eda.py` writes the figures and `docs/DATA_REALITIES.md` memo.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'outputs').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.viz.goalmouth import ZONE_ORDER, plot_goalmouth_heatmap

DATA_PATH = ROOT / 'outputs' / 'statsbomb' / 'penalties_statsbomb_clean.parquet'
df = pd.read_parquet(DATA_PATH)
df.shape

In [ ]:
summary = {
    'penalties': len(df),
    'conversion_rate': df['outcome_bin'].mean(),
    'shooters': df['shooter_id'].nunique(),
    'resolved_keepers': df['keeper_id'].nunique(),
    'competitions': df['competition_name'].nunique(),
    'shootout_share': df['is_shootout'].mean(),
}
summary

In [ ]:
zone_counts = df['shot_zone'].value_counts().reindex(ZONE_ORDER, fill_value=0)
zone_conversion = df.groupby('shot_zone')['outcome_bin'].mean().reindex(ZONE_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_goalmouth_heatmap(zone_counts.to_dict(), ax=axes[0], title='Placement counts', cbar_label='Kicks')
plot_goalmouth_heatmap(zone_conversion.fillna(0).to_dict(), ax=axes[1], title='Conversion by zone', cmap='Greens', value_format='{:.1%}', cbar_label='Conversion')
fig.tight_layout()

In [ ]:
df.groupby('shot_zone')['outcome_bin'].agg(n='size', goals='sum', conversion='mean').sort_values('n', ascending=False)

In [ ]:
coverage_fields = ['shot_zone', 'keeper_id', 'keeper_dive_direction', 'gk_freeze_location_x', 'gk_freeze_location_y', 'shootout_kick_index']
pd.DataFrame({
    'field': coverage_fields,
    'non_null': [df[c].notna().sum() for c in coverage_fields],
    'coverage': [df[c].notna().mean() for c in coverage_fields],
})

In [ ]:
# Regenerate all Step 4 figures and docs/DATA_REALITIES.md
%run ../reports/step4_eda.py